[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/10_Nonlinear_Diver_Control.ipynb)

# DiveLab

## Notebook 10 — Nonlinear Diver Control

**Guiding question:** How well do our linear control ideas survive when we restore the nonlinear physics of diving?

So far, many control ideas were developed around linearized models.

Now we return to the nonlinear plant:

- Boyle's law;
- depth-dependent buoyancy;
- quadratic drag;
- actuator saturation;
- external disturbances.

This is the beginning of a true nonlinear DiveLab simulator.

## Learning objectives

By the end of this lab, you will be able to:

- build a nonlinear vertical diver model;
- include Boyle's law explicitly in buoyancy;
- include quadratic hydrodynamic drag;
- model BCD gas as a controlled state;
- simulate actuator saturation;
- compare local linear intuition with nonlinear behavior;
- test controller robustness to larger disturbances;
- understand why linearization is local.

# 1. Why return to the nonlinear model?

Linearization is extremely useful.

Near an equilibrium:

$$
\dot x \approx A\delta x + B\delta u.
$$

But a real diver is governed by nonlinear relationships.

For example:

$$
V_g(z)\propto \frac{1}{P(z)}
$$

and:

$$
F_D \propto v|v|.
$$

These effects become more important when the state moves far from the equilibrium.

So the question is:

> Does a controller designed from local linear intuition still work when the motion becomes larger?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2. Physical constants and conventions

We keep the same sign convention:

- depth $z>0$ downward;
- vertical velocity $v>0$ upward.

Therefore:

$$
\dot z=-v.
$$

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0

z_e = 20.0
v_e = 0.0

Cd = 0.9
A_drag = 0.7

# 3. Pressure with depth

Ambient pressure is:

$$
P(z)=P_0+\rho gz.
$$

In [ ]:
def pressure_at_depth(z):
    return P0 + rho * g * z

# 4. Boyle's law inside the plant

Let $V_s$ be the equivalent gas volume referenced to surface pressure.

Then the actual gas volume at depth is:

$$
V_g(z,V_s)
=
V_s\frac{P_0}{P(z)}.
$$

In [ ]:
def gas_volume_at_depth(z, surface_gas_volume):
    return surface_gas_volume * P0 / pressure_at_depth(z)

# 5. Build neutral buoyancy at the operating point

Choose an equilibrium gas quantity:

$$
V_{s,e}=5\ \text{L}.
$$

Then choose the fixed displaced volume so that:

$$
F_B(z_e,V_{s,e})=mg.
$$

In [ ]:
Vs_e = 0.005  # m^3 = 5 L

Vg_e = gas_volume_at_depth(z_e, Vs_e)
fixed_volume = mass / rho - Vg_e

print(f"Gas volume at equilibrium depth: {Vg_e*1000:.3f} L")
print(f"Fixed displaced volume: {fixed_volume*1000:.3f} L")

# 6. Nonlinear buoyancy

Total displaced volume is:

$$
V_{\text{tot}}=V_f+V_g(z,V_s).
$$

So:

$$
F_B(z,V_s)
=
\rho g\left(
V_f+V_s\frac{P_0}{P_0+\rho gz}
\right).
$$

In [ ]:
def buoyant_force(z, Vs):
    Vg = gas_volume_at_depth(z, Vs)
    return rho * g * (fixed_volume + Vg)

# 7. Nonlinear hydrodynamic drag

We use:

$$
F_D(v)
=
\frac12\rho C_DA\,v|v|.
$$

The sign is carried by $v|v|$, so drag always opposes motion.

In [ ]:
def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

# 8. Nonlinear vertical dynamics

The plant becomes:

$$
\dot z=-v
$$

$$
m\dot v
=
F_B(z,V_s)-mg-F_D(v)+d(t)
$$

$$
\dot V_s=u
$$

where:

- $u$ is the BCD gas command;
- $d(t)$ is an external vertical disturbance force.

In [ ]:
def nonlinear_acceleration(z, v, Vs, disturbance_force=0.0):
    return (
        buoyant_force(z, Vs)
        - mass * g
        - drag_force(v)
        + disturbance_force
    ) / mass

# 9. Control law

We use a simple feedback structure around the equilibrium:

$$
u
=
K_z(z-z_e)
-
K_vv
-
K_V(V_s-V_{s,e}).
$$

The first two terms react to depth and velocity.

The third term discourages the gas state from drifting too far from its equilibrium value.

In [ ]:
Kz = 8e-5
Kv = 8e-4
Kv_s = 0.25

def controller(z, v, Vs):
    return (
        Kz * (z - z_e)
        - Kv * v
        - Kv_s * (Vs - Vs_e)
    )

## Why include gas-state feedback?

Without a term on $V_s$, the controller may correct depth while slowly accumulating a gas offset.

That can change the effective equilibrium.

Adding feedback on $V_s$ helps regulate the actuator state as well as motion.

# 10. Actuator saturation

A real BCD cannot add or vent gas at an unlimited rate.

So we impose:

$$
|u|\le u_{\max}.
$$

In [ ]:
u_max = 0.00035  # m^3/s equivalent surface gas volume rate

# 11. Nonlinear simulator

In [ ]:
def simulate_nonlinear(
    z0,
    v0,
    Vs0=Vs_e,
    duration=40.0,
    dt=0.01,
    controller_fn=controller,
    disturbance_fn=None,
    u_limit=u_max,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u = np.zeros(n)
    a = np.zeros(n)
    d = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for k in range(n - 1):
        disturbance = 0.0 if disturbance_fn is None else disturbance_fn(t[k])
        d[k] = disturbance

        uk = controller_fn(z[k], v[k], Vs[k])
        uk = np.clip(uk, -u_limit, u_limit)
        u[k] = uk

        ak = nonlinear_acceleration(z[k], v[k], Vs[k], disturbance)
        a[k] = ak

        # Euler integration
        v[k + 1] = v[k] + ak * dt
        z[k + 1] = z[k] - v[k + 1] * dt
        Vs[k + 1] = max(Vs[k] + uk * dt, 0.0)

        if z[k + 1] <= 0:
            z[k + 1:] = 0.0
            v[k + 1:] = v[k + 1]
            Vs[k + 1:] = Vs[k + 1]
            u[k + 1:] = 0.0
            d[k + 1:] = 0.0
            break

    u[-1] = u[-2]
    a[-1] = a[-2]
    d[-1] = d[-2]

    return t, z, v, Vs, u, a, d

# 12. Small perturbation: local behavior

Start near the equilibrium:

$$
z(0)=20\ \text{m}
$$

$$
v(0)=0.05\ \text{m/s}.
$$

This is the regime where linear intuition should work best.

In [ ]:
t_small, z_small, v_small, Vs_small, u_small, a_small, d_small = simulate_nonlinear(
    z0=z_e,
    v0=0.05
)

In [ ]:
plt.plot(t_small, z_small)
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Nonlinear closed loop: small perturbation")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(t_small, v_small)
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Velocity after a small perturbation")
plt.grid(True)
plt.show()

If the controller was designed sensibly around the equilibrium, small disturbances should be regulated.

This is the regime where local linearization is most trustworthy.

# 13. Larger initial condition

Now move much farther from the operating point:

$$
z(0)=15\ \text{m}.
$$

This is 5 m shallower than the equilibrium.

In [ ]:
t_large, z_large, v_large, Vs_large, u_large, a_large, d_large = simulate_nonlinear(
    z0=15.0,
    v0=0.0
)

In [ ]:
plt.plot(t_large, z_large)
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Nonlinear closed loop: large depth error")
plt.grid(True)
plt.show()

Now several nonlinear effects matter more:

- Boyle expansion is stronger at shallower depth;
- buoyancy sensitivity differs from the local value at 20 m;
- control saturation may activate;
- drag becomes important as velocity grows.

The controller is no longer operating in the exact regime for which the linear model was derived.

# 14. Compare small and large disturbances

In [ ]:
plt.plot(t_small, z_small - z_e, label="Small perturbation")
plt.plot(t_large, z_large - z_e, label="Large perturbation")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth error [m]")
plt.title("Local vs nonlinear closed-loop behavior")
plt.grid(True)
plt.legend()
plt.show()

This comparison illustrates a central idea:

> **linear control design is local.**

A controller that behaves well near the equilibrium may respond differently when the system moves far away.

# 15. Inspect control saturation

Let's examine whether the actuator hits its limits.

In [ ]:
plt.plot(t_large, u_large)
plt.axhline(u_max, linestyle="--")
plt.axhline(-u_max, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("Actuator saturation during large correction")
plt.grid(True)
plt.show()

When saturation occurs, the actual plant no longer follows the unconstrained control law.

This is another reason the real closed loop can differ strongly from linear predictions.

# 16. Gas-state evolution

The actuator changes the equivalent surface gas volume:

$$
V_s(t).
$$

In [ ]:
plt.plot(t_large, Vs_large * 1000)
plt.axhline(Vs_e * 1000, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Equivalent surface gas volume [L]")
plt.title("BCD gas state during nonlinear control")
plt.grid(True)
plt.show()

The BCD gas state is part of the dynamics.

Depth correction is achieved by changing buoyancy, not by commanding depth directly.

# 17. Add an external disturbance

We now apply a temporary upward force disturbance.

This can represent, in a highly simplified way:

- finning;
- a transient body-motion effect;
- an external hydrodynamic disturbance.

In [ ]:
def upward_push(t):
    if 10.0 <= t <= 12.0:
        return 25.0  # N upward
    return 0.0

In [ ]:
t_d, z_d, v_d, Vs_d, u_d, a_d, force_d = simulate_nonlinear(
    z0=z_e,
    v0=0.0,
    disturbance_fn=upward_push
)

In [ ]:
plt.plot(t_d, z_d)
plt.axhline(z_e, linestyle="--")
plt.axvspan(10, 12, alpha=0.15)

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Response to a temporary upward disturbance")
plt.grid(True)
plt.show()

The disturbance pushes the system away from equilibrium.

The controller must:

1. detect the resulting state error;
2. change gas volume;
3. change buoyancy;
4. oppose the disturbance-induced motion.

# 18. Disturbance rejection

In control theory, this is called **disturbance rejection**.

A good regulator should not only stabilize the equilibrium.

It should also recover after external perturbations.

In [ ]:
plt.plot(t_d, force_d)

plt.xlabel("Time [s]")
plt.ylabel("Disturbance force [N]")
plt.title("Applied disturbance")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(t_d, u_d)

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("Control reaction to disturbance")
plt.grid(True)
plt.show()

# 19. Nonlinear buoyancy sensitivity

The buoyancy-depth gain is not constant.

We can compute:

$$
\frac{\partial F_B}{\partial z}.
$$

For fixed $V_s$:

$$
\frac{\partial F_B}{\partial z}
=
-\rho g V_s P_0
\frac{\rho g}{(P_0+\rho gz)^2}.
$$

In [ ]:
def dFb_dz(z, Vs=Vs_e):
    return (
        -rho * g
        * Vs * P0
        * rho * g
        / pressure_at_depth(z)**2
    )

depth_grid = np.linspace(0, 40, 300)
sens = dFb_dz(depth_grid)

plt.plot(depth_grid, sens)

plt.xlabel("Depth [m]")
plt.ylabel("∂F_B / ∂z [N/m]")
plt.title("Nonlinear buoyancy sensitivity")
plt.grid(True)
plt.show()

The magnitude is largest near the surface.

So the same controller gain does not correspond to the same effective closed-loop dynamics at every depth.

This is a direct nonlinear limitation of one fixed linear controller.

# 20. Local linear models at different depths

We can compute a local linear coefficient:

$$
a_z(z)
=
\frac{1}{m}
\frac{\partial F_B}{\partial z}.
$$

In [ ]:
for depth in [5, 10, 20, 30, 40]:
    local_az = dFb_dz(depth) / mass
    print(f"{depth:2d} m → a_z = {local_az:.5f} 1/s²")

This suggests an important concept:

> the plant itself changes with operating point.

A controller tuned at 20 m may not have exactly the same behavior at 5 m or 40 m.

# 21. Gain scheduling preview

One practical nonlinear-control idea is **gain scheduling**.

Instead of using one constant controller:

$$
K=K_e,
$$

we allow the gains to depend on operating point:

$$
K=K(z).
$$

For example, the controller may use different gains near the surface, where buoyancy sensitivity is higher.

We will not fully design a gain-scheduled controller here, but the nonlinear plant now makes the motivation clear.

# 22. Compare different equilibrium depths

We can ask:

> What if we rebuild neutral buoyancy at another operating depth?

The same physical diver can have different local dynamics because Boyle compression changes with depth.

In [ ]:
def local_instability_rate(depth, Vs=Vs_e):
    az = dFb_dz(depth, Vs) / mass
    # For the two-state open-loop local model:
    # lambda = ±sqrt(-a_z)
    return np.sqrt(max(-az, 0.0))

for depth in [5, 10, 20, 30, 40]:
    rate = local_instability_rate(depth)
    print(f"{depth:2d} m → local open-loop instability rate ≈ {rate:.4f} 1/s")

The shallower operating region has stronger buoyancy sensitivity and therefore a faster local instability mechanism in this model.

This reconnects directly with the "first meters matter most" insight from Notebook 01 and Notebook 02.

# 23. A more realistic disturbance: periodic breathing

Breathing changes lung volume and therefore buoyancy.

We can represent this as a periodic disturbance force:

$$
d(t)=D\sin(2\pi ft).
$$

This is still a simplification, but it introduces a recurring disturbance.

In [ ]:
def breathing_disturbance(t):
    amplitude = 8.0   # N
    frequency = 0.25  # Hz
    return amplitude * np.sin(2 * np.pi * frequency * t)

In [ ]:
t_b, z_b, v_b, Vs_b, u_b, a_b, d_b = simulate_nonlinear(
    z0=z_e,
    v0=0.0,
    disturbance_fn=breathing_disturbance
)

In [ ]:
plt.plot(t_b, z_b)
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Closed-loop response to periodic breathing disturbance")
plt.grid(True)
plt.show()

The controller now responds to a periodic disturbance.

This lets us begin thinking in frequency-response terms:

> Which disturbances should the controller reject, and which should it avoid chasing?

# 24. Control effort under breathing disturbance

In [ ]:
plt.plot(t_b, u_b)

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("Control effort under periodic disturbance")
plt.grid(True)
plt.show()

A controller that reacts too strongly to every periodic fluctuation may waste actuator effort.

This connects control design with bandwidth and disturbance filtering.

# 25. Why a realistic simulator matters

The nonlinear simulator now contains:

- depth-dependent pressure;
- Boyle compression and expansion;
- nonlinear buoyancy;
- quadratic drag;
- dynamic gas state;
- actuator saturation;
- disturbances.

This is much closer to a plant model than the earlier linear examples.

But it is still simplified.

A more complete simulator might also include:

- lung volume as an explicit state;
- drysuit gas;
- body orientation;
- drag area changing with posture;
- valve dynamics;
- sensor delay and noise;
- controller delay;
- current and wave disturbances.

# 26. Linear theory is still valuable

Returning to the nonlinear model does not invalidate the previous notebooks.

Linear theory remains useful because it provides:

- local stability tests;
- eigenvalues and modes;
- observability;
- controllability;
- state-feedback design;
- Kalman filtering;
- LQG architecture.

The nonlinear simulator tells us where those approximations begin to break down.

# Exercises

### 1. Larger shallow disturbance

Try:

```python
z0 = 10.0
```

with the same controller.

Does the response remain well behaved?

### 2. Deeper initial condition

Try:

```python
z0 = 30.0
```

Compare the response with the shallow case.

### 3. Reduce actuator authority

Set:

```python
u_max = 0.0001
```

What happens after a large disturbance?

### 4. Increase drag area

Try:

```python
A_drag = 1.0
```

and:

```python
A_drag = 0.4
```

How does the response change?

### 5. Stronger breathing disturbance

Increase the disturbance amplitude.

At what point does the controller begin to saturate frequently?

# Challenge — compare local linear and nonlinear predictions

At several depths:

1. compute the local linear coefficient $a_z$;
2. derive the local open-loop eigenvalues;
3. simulate a very small nonlinear perturbation;
4. estimate the initial growth rate from the nonlinear simulation.

How well does the linear model predict the nonlinear plant locally?

In [ ]:
# Your code here

# Summary

In this notebook we returned to the nonlinear physical diver.

We restored:

- Boyle's law;
- nonlinear buoyancy;
- quadratic drag;
- BCD gas dynamics;
- saturation;
- disturbances.

We learned that:

- linearization is local;
- plant sensitivity changes with depth;
- large disturbances can activate nonlinearities and saturation;
- disturbance rejection is a core controller task;
- shallow-water dynamics are more sensitive in this model;
- breathing can be viewed as a periodic disturbance;
- gain scheduling becomes a natural idea when dynamics vary with operating point.

### Core insight

> **Linear control theory explains the neighborhood of an operating point. The nonlinear simulator tells us what happens when the diver leaves that neighborhood.**

### Next

Notebook 11 can introduce **trajectory tracking** rather than simple depth holding:

> How can the diver follow a commanded ascent profile while respecting velocity limits and maintaining stability?

That would move DiveLab from regulation to mission-level control.